In [32]:
import requests
from collections import defaultdict

GRAPHQL_URL = "https://api.morpho.org/graphql" # replace with actual endpoint

def fetch_markets_with_borrow():
    all_markets = []
    skip = 0
    page_size = 50

    while True:
        query = """
        query Markets($first: Int, $skip: Int) {
          markets(first: $first, skip: $skip, where: { chainId_in: [1] }) {
            items {
              uniqueKey
              loanAsset { symbol decimals }
              collateralAsset { symbol }
              historicalState {
                borrowAssets(options:  {
                    interval: WEEK
                }) {
                  x
                  y
                }
              }
            }
          }
        }
        """
        variables = {"first": page_size, "skip": skip}
        response = requests.post(GRAPHQL_URL, json={"query": query, "variables": variables})
        data = response.json()
        markets = data.get("data", {}).get("markets", {}).get("items", [])
        if not markets:
            break
        all_markets.extend(markets)
        skip += page_size
        print(f"Fetched {skip // page_size} pages")
        # break

    return all_markets

def process_markets(markets):
    results = []
    for market in markets:
        borrow_assets = market.get("historicalState", {}).get("borrowAssets", [])
        # y values are strings? Convert to float.
        max_borrow = max((float(point["y"]) for point in borrow_assets), default=0.0)
        results.append({
            "uniqueKey": market["uniqueKey"],
            "loanAsset": market["loanAsset"]["symbol"],
            "collateralAsset": market["collateralAsset"]["symbol"],
            "maxBorrow": max_borrow / 1e18,
            "tss_cnt": len(borrow_assets)
        })
    return results

def main():
    markets = fetch_markets_with_borrow()
    return markets
    # processed = process_markets(markets)
    # sorted_markets = sorted(processed, key=lambda x: x["maxBorrow"], reverse=True)

    # for m in sorted_markets:
    #     print(f"{m['loanAsset']}/{m['collateralAsset']} (key: {m['uniqueKey']}) max borrow: {m['maxBorrow']:.2f}, {m['tss_cnt']}")

    # return sorted_markets

if __name__ == "__main__":
    all_markets = main()

Fetched 1 pages
Fetched 2 pages
Fetched 3 pages
Fetched 4 pages
Fetched 5 pages
Fetched 6 pages
Fetched 7 pages
Fetched 8 pages
Fetched 9 pages
Fetched 10 pages
Fetched 11 pages
Fetched 12 pages
Fetched 13 pages
Fetched 14 pages
Fetched 15 pages
Fetched 16 pages
Fetched 17 pages
Fetched 18 pages
Fetched 19 pages
Fetched 20 pages
Fetched 21 pages
Fetched 22 pages
Fetched 23 pages
Fetched 24 pages
Fetched 25 pages


In [33]:
for i in all_markets:
    if i["uniqueKey"] == "0x64d65c9a2d91c36d56fbc42d69e979335320169b3df63bf92789e2c8883fcc64":
        print(i)

{'uniqueKey': '0x64d65c9a2d91c36d56fbc42d69e979335320169b3df63bf92789e2c8883fcc64', 'loanAsset': {'symbol': 'USDC', 'decimals': 6}, 'collateralAsset': {'symbol': 'cbBTC'}, 'historicalState': {'borrowAssets': [{'x': 1773178787, 'y': 370726341090007}, {'x': 1772668800, 'y': 382179343174153}, {'x': 1772064000, 'y': 395075865757703}, {'x': 1771459200, 'y': 436920384380085}, {'x': 1770854400, 'y': 433815859162042}, {'x': 1770249600, 'y': 340058362541958}, {'x': 1769644800, 'y': 458818337220294}, {'x': 1769040000, 'y': 455535927027030}, {'x': 1768435200, 'y': 431580836793973}, {'x': 1767830400, 'y': 400206174624287}, {'x': 1767225600, 'y': 405377040942864}, {'x': 1766620800, 'y': 374728540435348}, {'x': 1766016000, 'y': 391877485403403}, {'x': 1765411200, 'y': 405706354254248}, {'x': 1764806400, 'y': 412614658303182}, {'x': 1764201600, 'y': 421655287276877}, {'x': 1763596800, 'y': 407508361803758}, {'x': 1762992000, 'y': 432110742463188}, {'x': 1762387200, 'y': 430963410660172}, {'x': 176178

In [35]:
len(all_markets)
all_markets[0]
markets_prep = []
for market in all_markets:
    borrow_assets = market.get("historicalState", {}).get("borrowAssets", [])
        # y values are strings? Convert to float.
    max_borrow = max((float(point["y"]) for point in borrow_assets), default=0.0)
    try:
        markets_prep.append({
            "uniqueKey": market["uniqueKey"],
            "code": market["collateralAsset"]["symbol"] + "/" + market["loanAsset"]["symbol"],
            # "loanAsset": market["loanAsset"]["symbol"],
            # "collateralAsset": market["collateralAsset"]["symbol"],
            "maxBorrow": max_borrow / (10 ** (market["loanAsset"]["decimals"])),
        })
    except Exception as e:
        continue
markets_prep = sorted(markets_prep, key=lambda x: x["maxBorrow"], reverse=True)
len(markets_prep)

1188

In [75]:
markets_prep[:10]

[{'uniqueKey': '0x1dca6989b0d2b0a546530b3a739e91402eee2e1536a2d3ded4f5ce589a9cd1c2',
  'code': 'BONDUSD/USR',
  'maxBorrow': 11429545889.99309},
 {'uniqueKey': '0xa458018cf1a6e77ebbcc40ba5776ac7990e523b7cc5d0c1e740a4bbc13190d8f',
  'code': 'PT-USDS-14AUG2025/DAI',
  'maxBorrow': 462207277.431082},
 {'uniqueKey': '0x64d65c9a2d91c36d56fbc42d69e979335320169b3df63bf92789e2c8883fcc64',
  'code': 'cbBTC/USDC',
  'maxBorrow': 458818337.220294},
 {'uniqueKey': '0x45d97c66db5e803b9446802702f087d4293a2f74b370105dc3a88a278bf6bb21',
  'code': 'PT-USDe-25SEP2025/DAI',
  'maxBorrow': 452132525.75834537},
 {'uniqueKey': '0x5e3e6b1e01c5708055548d82d01db741e37d03b948a7ef9f3d4b962648bcbfa7',
  'code': 'PT-sUSDE-27MAR2025/DAI',
  'maxBorrow': 363268463.5545935},
 {'uniqueKey': '0x7a5d67805cb78fad2596899e0c83719ba89df353b931582eb7d3041fd5a06dc8',
  'code': 'PT-USDe-25SEP2025/USDC',
  'maxBorrow': 330973931.915632},
 {'uniqueKey': '0x3274643db77a064abd3bc851de77556a4ad2e2f502f4f0c80845fa8f909ecf0b',
  'cod

In [73]:
pt_markets = []
yield_tokens = []
pt_market_size = {}
for market in markets_prep:
    asset = market["code"].split("/")[0]
    if "usd0" in asset.lower():
        print(market)
    if 'PT-' in asset:
        pt_markets.append(market)
        yield_tokens.append(asset.split("-")[1])
        if yield_tokens[-1] in pt_market_size:
            pt_market_size[yield_tokens[-1]] = max(pt_market_size[yield_tokens[-1]], market["maxBorrow"])
        else:
            pt_market_size[yield_tokens[-1]] = market["maxBorrow"]
yield_tokens = list(set(yield_tokens))
len(pt_markets), len(yield_tokens)

{'uniqueKey': '0x1eda1b67414336cab3914316cb58339ddaef9e43f939af1fed162a989c98bc20', 'code': 'USD0++/USDC', 'maxBorrow': 252518126.546969}
{'uniqueKey': '0xb48bb53f0f2690c71e8813f2dc7ed6fca9ac4b0ace3faa37b4a8e5ece38fa1a2', 'code': 'USD0++/USDC', 'maxBorrow': 231886419.26626}
{'uniqueKey': '0xa59b6c3c6d1df322195bfb48ddcdcca1a4c0890540e8ee75815765096c1e8971', 'code': 'USD0USD0++/USDC', 'maxBorrow': 110303030.738743}
{'uniqueKey': '0x864c9b82eb066ae2c038ba763dfc0221001e62fc40925530056349633eb0a259', 'code': 'USD0USD0++/USDC', 'maxBorrow': 94397387.259163}
{'uniqueKey': '0x8411eeb07c8e32de0b3784b6b967346a45593bfd8baeb291cc209dc195c7b3ad', 'code': 'PT-USD0++-27MAR2025/USDC', 'maxBorrow': 71031506.885819}
{'uniqueKey': '0x92e5fe774a581e52428b4f8d6a775f35619a2e0a184363ae123fae478056d1cd', 'code': 'USD0++/USDC', 'maxBorrow': 70605568.246608}
{'uniqueKey': '0x147977320f168afc651b7e5a1849cc1b1e64e329e1bf0212fa49dcb2856074a4', 'code': 'PT-USD0++-27MAR2025/USDC', 'maxBorrow': 17122092.345604}
{'uni

(318, 63)

In [76]:
yield_tokens

['jrUSDe',
 'stcUSD',
 'siUSD',
 'USDG',
 'agETH',
 'reUSD',
 'AIDaUSDC',
 'tUSDe',
 'savUSD',
 'yvCurve',
 'sw',
 'avUSD',
 'USDS',
 'LBTC',
 'cUSD',
 'weETH',
 'EBTC',
 'beraSTONE',
 'ezETH',
 'syrupUSDC',
 'iUSD',
 'cUSDO',
 'fxSAVE',
 'corn',
 'lvlUSD',
 'mMEV',
 'RLP',
 'PendleInfiniFi',
 'csUSDL',
 'SolvBTC.BBN',
 'tETH',
 'USD3',
 'sBOLD',
 'SY wstUSR',
 'staking',
 'LPT',
 'sUSDf',
 'mHYPER',
 'DETH',
 'ysUSDC',
 'eUSDE',
 'sUSDE',
 'srUSDe',
 'wsrUSD',
 'AIDaUSDT',
 'slvlUSD(lvlUSD)',
 'PT',
 'wstUSR',
 'USR',
 'DUSD',
 'mEDGE',
 'RUSD',
 'sNUSD',
 'alUSD',
 'rsETH',
 'sdeUSD',
 'USD0++',
 'pUSDe',
 'srNUSD',
 'USDe',
 'sYUSD',
 'slvlUSD',
 'mAPOLLO']

In [87]:
yt_markets = []
for yt in yield_tokens:
    for market in markets_prep:
        if market["code"].startswith(yt + "/"):
            yt_markets.append(market)
            # print(f'{market["code"]}: {market["maxBorrow"]}')
            # break
yt_markets = sorted(yt_markets, key=lambda x: x["maxBorrow"], reverse=True)
seen = []
for i in yt_markets:
    if i["maxBorrow"] < 2_000_000:
        continue
    code, stable = i["code"].split("/")[0], i["code"].split("/")[1]
    seen.append(code.lower())
    print(f""" "eth_{code.lower()}_{stable.lower()}": "{i["uniqueKey"]}",  # Max borrow {i["maxBorrow"]:,} """)
    # if i["code"].split("/")[0] not in seen:
    #     print(f'{i["code"].split("/")[0]} \t\t {i["code"]}: Max Borrow {i["maxBorrow"]:,}, {i["uniqueKey"]} PT Market size {pt_market_size[i["code"].split("/")[0]]:,}')
    #     seen.append(i["code"].split("/")[0])


 "eth_usd0++_usdc": "0x1eda1b67414336cab3914316cb58339ddaef9e43f939af1fed162a989c98bc20",  # Max borrow 252,518,126.546969 
 "eth_usd0++_usdc": "0xb48bb53f0f2690c71e8813f2dc7ed6fca9ac4b0ace3faa37b4a8e5ece38fa1a2",  # Max borrow 231,886,419.26626 
 "eth_usde_dai": "0xc581c5f70bd1afa283eed57d1418c6432cbff1d862f94eaf58fdd4e46afbb67f",  # Max borrow 134,201,486.79947482 
 "eth_rlp_usdc": "0xe1b65304edd8ceaea9b629df4c3c926a37d1216e27900505c04f14b2ed279f33",  # Max borrow 130,292,931.169511 
 "eth_mhyper_usdc": "0x95c28d447950ca6c8bbfd25fc05b80b1fd7a1cdd17a3610b4b3f1ffc8dc2e2ed",  # Max borrow 121,157,336.705373 
 "eth_sdeusd_usdc": "0x0f9563442d64ab3bd3bcb27058db0b0d4046a4c46f0acd811dacae9551d2b129",  # Max borrow 96,325,154.583152 
 "eth_usr_usdc": "0x8e7cc042d739a365c43d0a52d5f24160fa7ae9b7e7c9a479bd02a56041d4cf77",  # Max borrow 92,806,567.78819 
 "eth_wsrusd_usdc": "0x1590cb22d797e226df92ebc6e0153427e207299916e7e4e53461389ad68272fb",  # Max borrow 89,961,932.034185 
 "eth_siusd_usdc": "

In [90]:
for i in pt_markets:
    if i["maxBorrow"] < 2_000_000:
        continue
    code, stable = i["code"].split("/")[0], i["code"].split("/")[1]
    flag = code.split('-')[1].lower() in seen
    if flag:
        print(f""" "eth_{code}_{stable.lower()}": "{i["uniqueKey"]}",  # Max borrow {i["maxBorrow"]:,} {flag, code.split('-')[1].lower() }""")


 "eth_PT-USDe-25SEP2025_dai": "0x45d97c66db5e803b9446802702f087d4293a2f74b370105dc3a88a278bf6bb21",  # Max borrow 452,132,525.75834537 (True, 'usde')
 "eth_PT-USDe-25SEP2025_usdc": "0x7a5d67805cb78fad2596899e0c83719ba89df353b931582eb7d3041fd5a06dc8",  # Max borrow 330,973,931.915632 (True, 'usde')
 "eth_PT-USDe-27NOV2025_usds": "0x8cdb63a27a48ac27fadc0f158a732104bcc4e10bb61c9a5095ea7c127204e26c",  # Max borrow 243,526,360.43353495 (True, 'usde')
 "eth_PT-USDe-27MAR2025_dai": "0xab0dcab71e65c05b7f241ea79a33452c87e62db387129e4abe15e458d433e4d8",  # Max borrow 97,377,815.0482268 (True, 'usde')
 "eth_PT-USD0++-27MAR2025_usdc": "0x8411eeb07c8e32de0b3784b6b967346a45593bfd8baeb291cc209dc195c7b3ad",  # Max borrow 71,031,506.885819 (True, 'usd0++')
 "eth_PT-syrupUSDC-28AUG2025_usdc": "0xa3819a7d2aee958ca0e7404137d012b51ea47d051db69d94656956eff8c80c23",  # Max borrow 57,772,451.663415 (True, 'syrupusdc')
 "eth_PT-USDe-25SEP2025_usdt": "0xb0a9ac81a8c6a5274aa1a8337aed35a2cb2cd4feb5c6d3b39d41f234fb